# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display


In [11]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

openai = OpenAI()

API key looks good so far


In [15]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
system_prompt = """
You are a patient programming tutor for a student taking an LLM engineering course.
They may be new to Python, APIs, and machine learning.

When they paste code or ask a technical question:
- Explain what it does in plain language first
- Then walk through it piece by piece
- Say why it is written that way, not only what it does
- Mention common mistakes or gotchas
- Use simple examples when they help
- Use markdown (headings, bullets, short code blocks)
- Keep the answer practical and concise, not academic
"""
def get_user_prompt(question):
    user_prompt = f"""
Answer this technical question for a student.
1. Short summary (2–3 sentences)
2. Step-by-step explanation
3. Tiny example
4. Why this pattern is useful
5. Common mistakes
Question:
{question}
"""
    return user_prompt



In [16]:
# Get gpt-4o-mini to answer, with streaming
def get_answer(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(question)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

get_answer(question)        

## Short Summary
The code is using a generator expression to yield the unique authors from a list of book dictionaries. It filters out any entries that do not have an author and returns the authors one by one.

## Step-by-Step Explanation
1. **`{book.get("author") for book in books if book.get("author")}`**: This part is a set comprehension that creates a set of authors.
   - `book.get("author")`: This retrieves the value associated with the key `"author"` from each `book` dictionary.
   - `for book in books`: This iterates over each book dictionary in the `books` list.
   - `if book.get("author")`: This checks if the author exists (is not `None` or an empty string). If there's no author, that book is ignored.
  
2. **`yield from`**: This keyword turns the set of authors into a generator, allowing you to yield each author one at a time. This is useful if you're going to iterate over them in a loop or send them one-by-one as output.

## Tiny Example
Here's a small example of how this works:

```python
books = [
    {"title": "Book A", "author": "Author A"},
    {"title": "Book B", "author": "Author B"},
    {"title": "Book C"},  # No author
    {"title": "Book D", "author": "Author A"},
]

def get_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in get_authors(books):
    print(author)
```
**Output:**
```
Author A
Author B
```
This yields the unique authors "Author A" and "Author B".

## Why This Pattern is Useful
- **Efficiency**: Using a generator (via `yield from`) is memory efficient because it produces values one at a time instead of creating a full list or set in memory.
- **Uniqueness**: Using a set comprehension automatically handles duplicates, ensuring each author is listed only once.

## Common Mistakes
- **Not checking for key existence**: If you try to access `book["author"]` instead of `book.get("author")`, it could raise a KeyError if "author" isn’t present in some dictionaries.
- **Confusing `yield` with `return`**: Remember that `yield` makes a function a generator. Using `return` would stop the function immediately and won't produce a series of values.
- **Forgetting to handle empty results**: If all books lack an author, the generator will yield no values, so ensure that your loop handles that gracefully.

In [17]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
def get_answer(question):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(question)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

get_answer(question)   

**Short Summary**
This code uses a feature of the `yield from` expression to iterate over a list of objects, where each object is itself an iterable. It retrieves the author of each book, which are then yielded by other expressions.

**Step-by-Step Explanation**

Here's a step-by-step breakdown:

1. `yield from` is a built-in Python feature that allows us to chain multiple generators together. It's like a pipe that collects the output of one generator and passes it to the next one.

2. Inside the outer loop `{ ... }`, we have a generator expression: `book.get("author") for book in books if book.get("author")`. This uses the `get()` method to retrieve the value of "author" for each book. The `for` clause iterates over the list of books (`books`).

3. The `if` clause inside the generator expression checks whether it should proceed. If the book has an "author" field, it proceeds; otherwise, it yields the result.

4. The `yield from` statement at the end of the outer loop resumes the execution at the point after the generator expression. Each subsequent expression in the list will be evaluated, but their return values will be collected and returned one by one.

5. The `*` operator unpacks the result of the generator expression into separate arguments.

**Tiny Example**

Using the imaginary list of books:
```python
books = [{"name": "Book1", "author": "John"}, {"name": "Book2", "author": "Jane"}]
```
The code would execute this:
```
# outer loop
for book in books:
    if book.get("author"):
        print(book["author"])
    yield book["author"]
```
Output: `John`, `Jane`

**Why this pattern is useful**

This pattern is useful when working with complex data structures like lists, dictionaries, or custom objects. It allows you to:

* Handle collections of objects in a pipeline-like fashion
* Extract and yield specific values from each object
* Use generators to handle large datasets efficiently

**Common Mistakes**

* Not using the `yield from` feature correctly (e.g., missing the `yield` statement)
* Using a generator expression inside a `for` loop
* Not collecting the return values of subsequent generators

**Best Practices**

* Use the `yield from` feature consistently throughout your code
* Remember to use the `*` operator to unpack generator expression results
* Avoid using `for` loops for complex data structures; use pipeline-like approaches instead.